In [2]:
import pandas as pd
df = pd.read_csv("../data/clean/merged_data_sets.csv")

C:\Users\franc\AppData\Local\Temp\ipykernel_19340\546622613.py:2: DtypeWarning: Columns (2: Variation) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/clean/merged_data_sets.csv")


In [3]:
df = df.drop(columns=["Unnamed: 0"])

In [4]:
df = df.drop_duplicates()

In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df_test = df[df["Variation"] == "Test"].copy()

In [7]:
df_control = df[df["Variation"] == "Control"].copy()

In [8]:
df_test = df_test[["client_id","process_step", "date_time"]]

In [9]:
df_control = df_control[["client_id","process_step", "date_time"]]

In [10]:
print("Test clients:", df_test["client_id"].nunique())
print("Control clients:", df_control["client_id"].nunique())

Test clients: 26968
Control clients: 23532


In [11]:
step_rank = {"start": 0, "step_1": 1, "step_2": 2, "step_3": 3, "confirm": 4}
rank_to_step = {v: k for k, v in step_rank.items()}

def abandoned_clients(df_group, group_name):
    d = df_group.copy()
    d["timestamp"] = pd.to_datetime(d["date_time"])
    d = d.sort_values(["client_id", "timestamp"], kind="stable").reset_index(drop=True)
    d["rank"] = d["process_step"].map(step_rank)

    unknown = d.loc[d["rank"].isna(), "process_step"].unique()
    if len(unknown):
        print(f"{group_name}: unexpected step names found (ignored in the order check): {unknown}")

    # out-of-order rule: jumping more than one step beyond the highest step reached so far
    d["max_before"] = d.groupby("client_id")["rank"].cummax().groupby(d["client_id"]).shift(1).fillna(-1)
    d["violation"] = d["rank"] > d["max_before"] + 1
    d["is_confirm"] = d["process_step"].eq("confirm")

    per_client = d.groupby("client_id").agg(
        n_events=("process_step", "size"),
        first_timestamp=("timestamp", "min"),
        last_timestamp=("timestamp", "max"),
        last_step=("process_step", "last"),
        furthest_rank=("rank", "max"),
        has_violation=("violation", "any"),
        has_confirm=("is_confirm", "any"),
    )
    per_client["furthest_step"] = per_client["furthest_rank"].map(rank_to_step)

    never_confirmed = ~per_client["has_confirm"]
    abandoned = never_confirmed & ~per_client["has_violation"]

    print(f"{group_name}: never confirmed = {never_confirmed.sum()} | "
          f"abandoned in order = {abandoned.sum()} | never confirmed and out of order = {(never_confirmed & per_client['has_violation']).sum()}")

    out = per_client[abandoned].reset_index()
    out["total_time"] = out["last_timestamp"] - out["first_timestamp"]
    out = out[["client_id", "last_step", "furthest_step", "n_events",
               "first_timestamp", "last_timestamp", "total_time"]]
    out.insert(0, "group", group_name)
    return out.reset_index(drop=True)

abandoned_test = abandoned_clients(df_test, "Test")
abandoned_control = abandoned_clients(df_control, "Control")

total_clients = {
    "Test": df_test["client_id"].nunique(),
    "Control": df_control["client_id"].nunique(),
}

print()
for name, t in [("Test", abandoned_test), ("Control", abandoned_control)]:
    print(f"{name}: {len(t)} of {total_clients[name]} clients "
          f"({len(t) / total_clients[name] * 100:.2f}%) abandoned in the middle")

Test: never confirmed = 8281 | abandoned in order = 8251 | never confirmed and out of order = 30
Control: never confirmed = 8098 | abandoned in order = 8062 | never confirmed and out of order = 36

Test: 8251 of 26968 clients (30.60%) abandoned in the middle
Control: 8062 of 23532 clients (34.26%) abandoned in the middle


In [12]:
abandoned_all = pd.concat([abandoned_test, abandoned_control], ignore_index=True)

def drop_off_summary(col):
    s = abandoned_all.groupby(["group", col]).size().rename("clients").reset_index()
    s["pct_of_group_clients"] = (s["clients"] / s["group"].map(total_clients) * 100).round(2)
    s["pct_of_abandoned"] = (s["clients"] / s.groupby("group")["clients"].transform("sum") * 100).round(2)
    s["order"] = s[col].map(step_rank)
    return s.sort_values(["group", "order"]).drop(columns="order").reset_index(drop=True)

print("BY LAST STEP (where the client was when they left)")
display(drop_off_summary("last_step"))

print("BY FURTHEST STEP (the furthest step they reached)")
display(drop_off_summary("furthest_step"))

print("TEST - abandoned in the middle")
display(abandoned_test)

print("CONTROL - abandoned in the middle")
display(abandoned_control)

BY LAST STEP (where the client was when they left)


,group,last_step,clients,pct_of_group_clients,pct_of_abandoned
0,Control,start,4900,20.82,60.78
1,Control,step_1,1224,5.20,15.18
2,Control,step_2,727,3.09,9.02
3,Control,step_3,1211,5.15,15.02
4,Test,start,5336,19.79,64.67
5,Test,step_1,1356,5.03,16.43
6,Test,step_2,567,2.10,6.87
7,Test,step_3,992,3.68,12.02


BY FURTHEST STEP (the furthest step they reached)


,group,furthest_step,clients,pct_of_group_clients,pct_of_abandoned
0,Control,start,3285,13.96,40.75
1,Control,step_1,1451,6.17,18.00
2,Control,step_2,1255,5.33,15.57
3,Control,step_3,2071,8.80,25.69
4,Test,start,2454,9.10,29.74
5,Test,step_1,1972,7.31,23.90
6,Test,step_2,1403,5.20,17.00
7,Test,step_3,2422,8.98,29.35


TEST - abandoned in the middle


,group,client_id,last_step,furthest_step,n_events,first_timestamp,last_timestamp,total_time
0,Test,934,start,start,4,2017-04-18 02:36:30,2017-04-18 02:38:52,0 days 00:02:22
1,Test,1346,step_3,step_3,10,2017-06-06 18:20:28,2017-06-06 18:28:43,0 days 00:08:15
2,Test,2720,start,start,3,2017-04-03 16:05:30,2017-04-03 16:08:23,0 days 00:02:53
3,Test,5845,step_3,step_3,9,2017-04-12 12:14:40,2017-05-11 12:28:02,29 days 00:13:22
4,Test,6022,start,step_1,3,2017-04-12 21:15:24,2017-04-12 21:21:47,0 days 00:06:23
...,...,...,...,...,...,...,...,...
8246,Test,9994633,start,start,1,2017-05-08 08:28:58,2017-05-08 08:28:58,0 days 00:00:00
8247,Test,9995295,start,step_3,9,2017-03-29 11:15:59,2017-03-29 11:18:52,0 days 00:02:53
8248,Test,9999150,start,step_1,3,2017-05-29 16:55:12,2017-05-29 16:55:30,0 days 00:00:18
8249,Test,9999626,step_1,step_1,2,2017-05-14 09:07:51,2017-05-14 09:07:59,0 days 00:00:08


CONTROL - abandoned in the middle


,group,client_id,last_step,furthest_step,n_events,first_timestamp,last_timestamp,total_time
0,Control,1028,step_1,step_3,9,2017-04-08 18:51:28,2017-04-08 19:00:26,0 days 00:08:58
1,Control,1104,start,start,2,2017-06-12 07:49:18,2017-06-20 22:31:33,8 days 14:42:15
2,Control,1186,step_2,step_2,4,2017-04-08 15:59:16,2017-04-08 18:05:24,0 days 02:06:08
3,Control,1368,start,start,1,2017-04-09 21:58:28,2017-04-09 21:58:28,0 days 00:00:00
4,Control,2439,start,start,1,2017-03-23 14:59:23,2017-03-23 14:59:23,0 days 00:00:00
...,...,...,...,...,...,...,...,...
8057,Control,9994194,start,start,1,2017-04-20 20:25:24,2017-04-20 20:25:24,0 days 00:00:00
8058,Control,9995265,step_3,step_3,7,2017-04-08 20:08:00,2017-04-08 20:12:37,0 days 00:04:37
8059,Control,9996877,start,start,1,2017-05-03 18:08:32,2017-05-03 18:08:32,0 days 00:00:00
8060,Control,9997391,step_3,step_3,8,2017-04-05 15:37:56,2017-04-05 15:41:39,0 days 00:03:43


In [13]:
total_clients

{'Test': 26968, 'Control': 23532}